# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema JSON-LD file](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and examine the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# View metadata
print(f"Dataset @id: {dataset.metadata.id}")
print(f"Name: {dataset.metadata.name}")
print(f"Version: {dataset.metadata.version}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Description: {dataset.metadata.description}")
print(f"Date published: {dataset.metadata.date_published}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Record sets (`cr:RecordSet`) define the main tables or logical groupings of records within the dataset.

In [ ]:
# List all available record sets and fields by their @id
print("Available record sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"  - @id: {rs.id}, name: {getattr(rs, 'name', None)}")
    if hasattr(rs, 'fields'):
        print("    Fields:")
        for field in rs.fields:
            print(f"      - field @id: {field.id}, name: {getattr(field, 'name', None)}, datatype: {getattr(field, 'data_type', None)}")
    if hasattr(rs, 'columns'):
        print("    Columns:")
        for col in rs.columns:
            print(f"      - column @id: {col.id}, name: {getattr(col, 'name', None)}, datatype: {getattr(col, 'data_type', None)}")
# Get an example of records from each record set
for rs in record_sets:
    print(f"\nExample record from record set @id: {rs.id}")
    records_iter = dataset.records(record_set=rs.id)
    try:
        example_record = next(records_iter)
        print(example_record)
    except StopIteration:
        print("No records found in this record set.")

## 3. Data Extraction
Load selected record sets and their records into Pandas DataFrames for further analysis. All entities are referenced by their Croissant `@id`.

In [ ]:
# List record set @ids from Step 2
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("No records found.")

# For subsequent analysis, choose the first record set with records
primary_record_set_id = next(iter(dataframes.keys()))
df = dataframes[primary_record_set_id]
print(f"\nWill use record set '@id': {primary_record_set_id} with columns: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Explore, filter, and transform the dataset. In this section, select numeric and categorical fields by their `@id` (column names correspond to their `@id`).

In [ ]:
# Identify a numeric field and a group field from record set columns:
print(f"Columns: {df.columns.tolist()}")

# --- Example: Choose a likely numeric field (by inspection):
# Substitute actual field @id as found in your dataset (output above). For illustration:
# Suppose 'cr:log_likelihood' is a numeric field and 'cr:ward' is a group/categorical field.
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if any(s in col.lower() for s in ['log_likelihood', 'score', 'value', 'coefficient', 'std', 'mean', 'error']):
        numeric_field_id = col
    if any(s in col.lower() for s in ['ward', 'county', 'group']):
        group_field_id = col

if numeric_field_id is not None:
    print(f"Using numeric field: {numeric_field_id}")
else:
    print("No obvious numeric field detected. Set 'numeric_field_id' manually.")

if group_field_id is not None:
    print(f"Using group field: {group_field_id}")

# Drop missing or non-numeric values for analysis
if numeric_field_id is not None:
    df_clean = df.copy()
    df_clean = df_clean[pd.to_numeric(df_clean[numeric_field_id], errors='coerce').notna()]
    df_clean[numeric_field_id] = pd.to_numeric(df_clean[numeric_field_id], errors='coerce')

    # Filter: keep records with value above a threshold (use mean or 10 as demonstration)
    threshold = df_clean[numeric_field_id].mean() if df_clean[numeric_field_id].mean() > 0 else 10
    filtered_df = df_clean[df_clean[numeric_field_id] > threshold]
    print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group, if possible
    if group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean {numeric_field_id} by {group_field_id}:")
        print(grouped.head())

## 5. Visualization
Visualize numerical distributions and grouped comparisons. This example uses matplotlib and seaborn. Please adjust field `@id`s and groups to fit your dataset contents.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_clean[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    if group_field_id in df_clean.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_clean)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a Croissant-described dataset with `mlcroissant`, using solely `@id` references for all dataset entities. We:
- Inspected dataset metadata and available record sets via their `@id`.
- Loaded and previewed DataFrames for each record set and referenced all columns using their `@id`.
- Performed simple filtering, normalization, and grouping operations.
- Visualized distribution and groupwise properties for selected numeric fields.

Further analyses may incorporate additional fields or advanced statistical methods as needed. Refer to the FAIR² dataset's Croissant schema and documentation for field meanings and usage guidance.